In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

import os
base = "/kaggle/input"
for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)
    if level < 4:
        print("  " * level + os.path.basename(root) + "/")

GPU available: True
GPU name: Tesla T4
input/
  datasets/
    prathamesh25022005/
      chest-xray-pneumonia-local/


In [2]:
dataset_path = "/kaggle/input/datasets/prathamesh25022005/chest-xray-pneumonia-local"
for item in os.listdir(dataset_path):
    print(item)

chest_xray


In [3]:
DATA_DIR = "/kaggle/input/datasets/prathamesh25022005/chest-xray-pneumonia-local/chest_xray"

for split in ["train", "val", "test"]:
    path = os.path.join(DATA_DIR, split)
    print(split, "->", os.listdir(path) if os.path.exists(path) else "NOT FOUND")

train -> ['PNEUMONIA', 'NORMAL']
val -> ['PNEUMONIA', 'NORMAL']
test -> ['PNEUMONIA', 'NORMAL']


In [4]:
DATA_DIR = "/kaggle/input/datasets/prathamesh25022005/chest-xray-pneumonia-local/chest_xray"

In [5]:
import os, time, copy, json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score
)
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/prathamesh25022005/chest-xray-pneumonia-local/chest_xray"
BATCH_SIZE = 32
NUM_EPOCHS = 10          
LEARNING_RATE = 1e-4
IMG_SIZE = 224
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR = "/kaggle/working/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")

Using device: cuda


In [6]:
train_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_tfms)
val_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform=eval_tfms)
test_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), transform=eval_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
classes = train_ds.classes

print("Classes:", classes)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Classes: ['NORMAL', 'PNEUMONIA']
Train: 5216 | Val: 16 | Test: 624


In [7]:
def build_model(name, num_classes=2):
    if name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "mobilenet_v2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"Unknown model {name}")
    return m.to(DEVICE)

In [8]:
def train_one_model(name, train_loader, val_loader):
    print(f"\n{'='*50}\nTraining {name}\n{'='*50}")
    model = build_model(name)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_val_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())
    history = []

    for epoch in range(NUM_EPOCHS):
        start = time.time()
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        model.eval()
        v_correct, v_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                v_correct += (outputs.argmax(1) == labels).sum().item()
                v_total += labels.size(0)
        val_acc = v_correct / v_total

        elapsed = time.time() - start
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | train_loss={train_loss:.4f} "
              f"train_acc={train_acc:.4f} val_acc={val_acc:.4f} ({elapsed:.1f}s)")
        history.append({"epoch": epoch+1, "train_loss": train_loss,
                         "train_acc": train_acc, "val_acc": val_acc})

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_weights)
    torch.save(model.state_dict(), os.path.join(RESULTS_DIR, f"{name}_best.pt"))
    pd.DataFrame(history).to_csv(os.path.join(RESULTS_DIR, f"{name}_history.csv"), index=False)
    return model

In [9]:
def evaluate_model(name, model, test_loader, classes):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    metrics = {
        "model": name,
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds),
        "recall": recall_score(all_labels, all_preds),
        "f1_score": f1_score(all_labels, all_preds),
        "auc_roc": roc_auc_score(all_labels, all_probs),
    }
    cm = confusion_matrix(all_labels, all_preds)
    print(f"\n{name} — Test Metrics:")
    for k, v in metrics.items():
        if k != "model":
            print(f"  {k}: {v:.4f}")
    print(f"  confusion_matrix (rows=true, cols=pred, order={classes}):\n{cm}")

    with open(os.path.join(RESULTS_DIR, f"{name}_confusion_matrix.json"), "w") as f:
        json.dump(cm.tolist(), f)
    return metrics

In [10]:
model_names = ["resnet18", "mobilenet_v2", "efficientnet_b0"]
all_metrics = []

for name in model_names:
    model = train_one_model(name, train_loader, val_loader)
    metrics = evaluate_model(name, model, test_loader, classes)
    all_metrics.append(metrics)

results_df = pd.DataFrame(all_metrics)
results_df.to_csv(os.path.join(RESULTS_DIR, "comparison_table.csv"), index=False)
print("\nComparison table:")
print(results_df.to_string(index=False))


Training resnet18
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 144MB/s]


Epoch 1/10 | train_loss=0.1142 train_acc=0.9542 val_acc=0.9375 (91.3s)
Epoch 2/10 | train_loss=0.0599 train_acc=0.9774 val_acc=0.9375 (76.1s)
Epoch 3/10 | train_loss=0.0398 train_acc=0.9856 val_acc=0.8125 (76.7s)
Epoch 4/10 | train_loss=0.0335 train_acc=0.9872 val_acc=0.7500 (77.5s)
Epoch 5/10 | train_loss=0.0283 train_acc=0.9906 val_acc=0.8125 (76.2s)
Epoch 6/10 | train_loss=0.0400 train_acc=0.9860 val_acc=1.0000 (77.1s)
Epoch 7/10 | train_loss=0.0177 train_acc=0.9937 val_acc=1.0000 (76.1s)
Epoch 8/10 | train_loss=0.0195 train_acc=0.9931 val_acc=0.6250 (75.5s)
Epoch 9/10 | train_loss=0.0193 train_acc=0.9950 val_acc=0.8750 (76.6s)
Epoch 10/10 | train_loss=0.0121 train_acc=0.9946 val_acc=0.6250 (76.2s)

resnet18 — Test Metrics:
  accuracy: 0.8782
  precision: 0.8384
  recall: 0.9974
  f1_score: 0.9110
  auc_roc: 0.9781
  confusion_matrix (rows=true, cols=pred, order=['NORMAL', 'PNEUMONIA']):
[[159  75]
 [  1 389]]

Training mobilenet_v2
Downloading: "https://download.pytorch.org/models/

100%|██████████| 13.6M/13.6M [00:00<00:00, 98.2MB/s]


Epoch 1/10 | train_loss=0.2220 train_acc=0.9233 val_acc=0.6875 (78.6s)
Epoch 2/10 | train_loss=0.0784 train_acc=0.9703 val_acc=0.6875 (78.9s)
Epoch 3/10 | train_loss=0.0572 train_acc=0.9803 val_acc=0.6875 (78.0s)
Epoch 4/10 | train_loss=0.0485 train_acc=0.9829 val_acc=0.8750 (79.3s)
Epoch 5/10 | train_loss=0.0451 train_acc=0.9835 val_acc=0.6875 (76.8s)
Epoch 6/10 | train_loss=0.0344 train_acc=0.9881 val_acc=1.0000 (77.6s)
Epoch 7/10 | train_loss=0.0254 train_acc=0.9925 val_acc=0.8125 (77.5s)
Epoch 8/10 | train_loss=0.0217 train_acc=0.9931 val_acc=0.8750 (77.8s)
Epoch 9/10 | train_loss=0.0238 train_acc=0.9921 val_acc=0.7500 (77.3s)
Epoch 10/10 | train_loss=0.0190 train_acc=0.9941 val_acc=0.7500 (76.9s)

mobilenet_v2 — Test Metrics:
  accuracy: 0.8558
  precision: 0.8151
  recall: 0.9949
  f1_score: 0.8961
  auc_roc: 0.9676
  confusion_matrix (rows=true, cols=pred, order=['NORMAL', 'PNEUMONIA']):
[[146  88]
 [  2 388]]

Training efficientnet_b0
Downloading: "https://download.pytorch.org/

100%|██████████| 20.5M/20.5M [00:00<00:00, 107MB/s] 


Epoch 1/10 | train_loss=0.1644 train_acc=0.9482 val_acc=0.7500 (79.1s)
Epoch 2/10 | train_loss=0.0675 train_acc=0.9772 val_acc=0.6875 (78.4s)
Epoch 3/10 | train_loss=0.0447 train_acc=0.9839 val_acc=0.6875 (77.6s)
Epoch 4/10 | train_loss=0.0359 train_acc=0.9866 val_acc=0.9375 (79.3s)
Epoch 5/10 | train_loss=0.0294 train_acc=0.9910 val_acc=0.9375 (77.7s)
Epoch 6/10 | train_loss=0.0242 train_acc=0.9906 val_acc=0.6875 (78.3s)
Epoch 7/10 | train_loss=0.0276 train_acc=0.9906 val_acc=1.0000 (78.4s)
Epoch 8/10 | train_loss=0.0174 train_acc=0.9941 val_acc=0.8750 (78.6s)
Epoch 9/10 | train_loss=0.0154 train_acc=0.9939 val_acc=0.9375 (78.4s)
Epoch 10/10 | train_loss=0.0175 train_acc=0.9937 val_acc=0.7500 (78.3s)

efficientnet_b0 — Test Metrics:
  accuracy: 0.8478
  precision: 0.8054
  recall: 0.9974
  f1_score: 0.8912
  auc_roc: 0.9514
  confusion_matrix (rows=true, cols=pred, order=['NORMAL', 'PNEUMONIA']):
[[140  94]
 [  1 389]]

Comparison table:
          model  accuracy  precision   recall  